In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings

warnings.filterwarnings('ignore')

# Paths
RAW = '../data/raw/'
OUTPUT = '../outputs/cleaned/'

os.makedirs(OUTPUT, exist_ok=True)

print("Libraries loaded ✓")

Libraries loaded ✓


In [3]:
orders       = pd.read_csv(RAW + 'olist_orders_dataset.csv')
customers    = pd.read_csv(RAW + 'olist_customers_dataset.csv')
order_items  = pd.read_csv(RAW + 'olist_order_items_dataset.csv')
payments     = pd.read_csv(RAW + 'olist_order_payments_dataset.csv')
reviews      = pd.read_csv(RAW + 'olist_order_reviews_dataset.csv')
products     = pd.read_csv(RAW + 'olist_products_dataset.csv')
sellers      = pd.read_csv(RAW + 'olist_sellers_dataset.csv')
geo          = pd.read_csv(RAW + 'olist_geolocation_dataset.csv')
category_map = pd.read_csv(RAW + 'product_category_name_translation.csv')

print("All tables loaded ✓")
print(f"\nRow counts:")
for name, df in {
    'orders': orders, 'customers': customers,
    'order_items': order_items, 'payments': payments,
    'reviews': reviews, 'products': products,
    'sellers': sellers, 'category_map': category_map
}.items():
    print(f"  {name:15s}: {len(df):,} rows")

All tables loaded ✓

Row counts:
  orders         : 99,441 rows
  customers      : 99,441 rows
  order_items    : 112,650 rows
  payments       : 103,886 rows
  reviews        : 99,224 rows
  products       : 32,951 rows
  sellers        : 3,095 rows
  category_map   : 71 rows


In [4]:
date_cols = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

for col in date_cols:
    orders[col] = pd.to_datetime(orders[col])

print("Dates parsed ✓")
print(f"\nDate range: {orders['order_purchase_timestamp'].min().date()} "
      f"→ {orders['order_purchase_timestamp'].max().date()}")

Dates parsed ✓

Date range: 2016-09-04 → 2018-10-17


In [6]:
print(f"Order statuses:\n{orders['order_status'].value_counts()}\n")

orders_delivered = orders[orders['order_status'] == 'delivered'].copy()

print(f"Total orders:    {len(orders):,}")
print(f"Delivered orders: {len(orders_delivered):,}")
print(f"Dropped:          {len(orders) - len(orders_delivered):,} "
      f"({(1 - len(orders_delivered)/len(orders))*100:.1f}%)")

Order statuses:
order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

Total orders:    99,441
Delivered orders: 96,478
Dropped:          2,963 (3.0%)


In [7]:
# Translate product categories to English
products_eng = products.merge(category_map, on='product_category_name', how='left')

# Join order items to products
items_products = order_items.merge(
    products_eng[['product_id', 'product_category_name_english']],
    on='product_id', how='left'
)

# Join to sellers
items_full = items_products.merge(
    sellers[['seller_id', 'seller_city', 'seller_state']],
    on='seller_id', how='left'
)

# Join orders to customers
orders_customers = orders_delivered.merge(
    customers[['customer_id', 'customer_city', 'customer_state']],
    on='customer_id', how='left'
)

# Master table — everything joined
master = orders_customers.merge(items_full, on='order_id', how='left')

# Add payments (aggregated per order first to avoid row duplication)
pay_agg = payments.groupby('order_id').agg(
    payment_value=('payment_value', 'sum'),
    payment_type=('payment_type', lambda x: x.mode()[0])
).reset_index()

master = master.merge(pay_agg, on='order_id', how='left')

# Add review scores
reviews_agg = reviews.groupby('order_id')['review_score'].mean().reset_index()
master = master.merge(reviews_agg, on='order_id', how='left')

print(f"Master table shape: {master.shape}")
print(f"Columns: {list(master.columns)}")

Master table shape: (110197, 22)
Columns: ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'customer_city', 'customer_state', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value', 'product_category_name_english', 'seller_city', 'seller_state', 'payment_value', 'payment_type', 'review_score']


In [8]:
# --- Delivery delay (days) ---
master['delivery_delay_days'] = (
    master['order_delivered_customer_date'] -
    master['order_estimated_delivery_date']
).dt.days

# Positive = late, Negative = early, Zero = on time
master['delivery_status'] = master['delivery_delay_days'].apply(
    lambda x: 'Late' if x > 0 else ('Early' if x < 0 else 'On Time')
)

# --- Actual delivery time (days from purchase to delivery) ---
master['actual_delivery_days'] = (
    master['order_delivered_customer_date'] -
    master['order_purchase_timestamp']
).dt.days

# --- Revenue columns ---
master['total_order_value'] = master['price'] + master['freight_value']

# --- Time dimensions (for Power BI time intelligence) ---
master['purchase_year']    = master['order_purchase_timestamp'].dt.year
master['purchase_month']   = master['order_purchase_timestamp'].dt.month
master['purchase_month_name'] = master['order_purchase_timestamp'].dt.strftime('%b')
master['purchase_quarter'] = master['order_purchase_timestamp'].dt.quarter
master['purchase_dayofweek'] = master['order_purchase_timestamp'].dt.day_name()

print("Feature engineering complete ✓")
print(f"\nDelivery status breakdown:")
print(master['delivery_status'].value_counts())
print(f"\nAverage actual delivery time: {master['actual_delivery_days'].mean():.1f} days")
print(f"Average delay (late orders):  {master[master['delivery_delay_days'] > 0]['delivery_delay_days'].mean():.1f} days")

Feature engineering complete ✓

Delivery status breakdown:
delivery_status
Early      101475
Late         7264
On Time      1458
Name: count, dtype: int64

Average actual delivery time: 12.0 days
Average delay (late orders):  10.5 days


In [9]:
import numpy as np

# RFM is calculated at customer level, not order-item level
# So we deduplicate to one row per order first
orders_level = master.drop_duplicates(subset='order_id').copy()

# Reference date — day after the last purchase in the dataset
reference_date = orders_level['order_purchase_timestamp'].max() + pd.Timedelta(days=1)

# --- Calculate R, F, M per customer ---
rfm = orders_level.groupby('customer_id').agg(
    Recency=('order_purchase_timestamp', lambda x: (reference_date - x.max()).days),
    Frequency=('order_id', 'count'),
    Monetary=('payment_value', 'sum')
).reset_index()

# --- Score each metric 1–5 (5 = best) ---
rfm['R_score'] = pd.qcut(rfm['Recency'], q=5, labels=[5,4,3,2,1]).astype(int)
rfm['F_score'] = pd.qcut(rfm['Frequency'].rank(method='first'), q=5, labels=[1,2,3,4,5]).astype(int)
rfm['M_score'] = pd.qcut(rfm['Monetary'], q=5, labels=[1,2,3,4,5]).astype(int)

# --- Combine into RFM score ---
rfm['RFM_score'] = rfm['R_score'].astype(str) + rfm['F_score'].astype(str) + rfm['M_score'].astype(str)
rfm['RFM_total'] = rfm['R_score'] + rfm['F_score'] + rfm['M_score']

# --- Segment customers ---
def segment(row):
    r, f, m = row['R_score'], row['F_score'], row['M_score']
    if r >= 4 and f >= 4 and m >= 4:
        return 'Champions'
    elif r >= 3 and f >= 3 and m >= 3:
        return 'Loyal Customers'
    elif r >= 4 and f <= 2:
        return 'New Customers'
    elif r <= 2 and f >= 3:
        return 'At Risk'
    elif r == 1 and f == 1:
        return 'Lost'
    else:
        return 'Potential Loyalists'

rfm['Segment'] = rfm.apply(segment, axis=1)

print("RFM scoring complete ✓")
print(f"\nCustomer segments:")
print(rfm['Segment'].value_counts())
print(f"\nRFM table preview:")
print(rfm.head())

RFM scoring complete ✓

Customer segments:
Segment
Potential Loyalists    32617
At Risk                23246
New Customers          15606
Loyal Customers        14871
Champions               6363
Lost                    3775
Name: count, dtype: int64

RFM table preview:
                        customer_id  Recency  Frequency  Monetary  R_score  \
0  00012a2ce6f8dcda20d059ce98491703      288          1    114.74        2   
1  000161a058600d5901f007fab4c27140      410          1     67.41        1   
2  0001fd6190edaaf884bcaf3d49edf079      548          1    195.42        1   
3  0002414f95344307404f0ace7a26f1d5      379          1    179.35        2   
4  000379cdec625522490c315e70c7a9fb      150          1    107.01        4   

   F_score  M_score RFM_score  RFM_total              Segment  
0        1        3       213          6  Potential Loyalists  
1        1        2       112          4                 Lost  
2        1        4       114          6                 Lost  
3   

In [10]:
# --- 1. Sales overview dataset ---
sales = master[[
    'order_id', 'order_purchase_timestamp', 'purchase_year',
    'purchase_month', 'purchase_month_name', 'purchase_quarter',
    'purchase_dayofweek', 'price', 'freight_value',
    'total_order_value', 'payment_type', 'payment_value',
    'customer_state', 'product_category_name_english'
]].copy()

sales.to_csv(OUTPUT + 'sales_overview.csv', index=False)
print(f"sales_overview.csv exported — {len(sales):,} rows ✓")

# --- 2. Delivery performance dataset ---
delivery = master[[
    'order_id', 'order_purchase_timestamp', 'purchase_year',
    'purchase_month', 'purchase_month_name',
    'order_estimated_delivery_date', 'order_delivered_customer_date',
    'actual_delivery_days', 'delivery_delay_days',
    'delivery_status', 'customer_state', 'seller_state',
    'review_score', 'price'
]].drop_duplicates(subset='order_id').copy()

delivery.to_csv(OUTPUT + 'delivery_performance.csv', index=False)
print(f"delivery_performance.csv exported — {len(delivery):,} rows ✓")

# --- 3. Customer RFM dataset ---
rfm.to_csv(OUTPUT + 'customer_rfm.csv', index=False)
print(f"customer_rfm.csv exported — {len(rfm):,} rows ✓")

# --- 4. Seller & product performance dataset ---
seller_product = master.groupby(
    ['seller_id', 'seller_state', 'product_category_name_english']
).agg(
    total_revenue=('price', 'sum'),
    total_orders=('order_id', 'nunique'),
    avg_review_score=('review_score', 'mean'),
    avg_freight=('freight_value', 'mean')
).reset_index()

seller_product.to_csv(OUTPUT + 'seller_product_performance.csv', index=False)
print(f"seller_product_performance.csv exported — {len(seller_product):,} rows ✓")

# --- 5. Monthly revenue summary ---
monthly = master.groupby(
    ['purchase_year', 'purchase_month', 'purchase_month_name']
).agg(
    total_revenue=('price', 'sum'),
    total_orders=('order_id', 'nunique'),
    avg_order_value=('payment_value', 'mean'),
    avg_review_score=('review_score', 'mean')
).reset_index().sort_values(['purchase_year', 'purchase_month'])

monthly.to_csv(OUTPUT + 'monthly_revenue.csv', index=False)
print(f"monthly_revenue.csv exported — {len(monthly):,} rows ✓")

print("\nAll datasets exported to outputs/cleaned/ ✓")
print("\nReady for Power BI 🎯")

sales_overview.csv exported — 110,197 rows ✓
delivery_performance.csv exported — 96,478 rows ✓
customer_rfm.csv exported — 96,478 rows ✓
seller_product_performance.csv exported — 6,144 rows ✓
monthly_revenue.csv exported — 23 rows ✓

All datasets exported to outputs/cleaned/ ✓

Ready for Power BI 🎯
